In [ ]:
# ============================================================
# ReGEN-TAD Ablation Study with Dedicated Variant Classes
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model, optimizers
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


# ============================================================
# 1. SHARED BACKBONE
# ============================================================

class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = np.arange(max_len)[:, None]
        i = np.arange(d_model)[None, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)
        angle = pos * angle_rates
        pe = np.zeros((max_len, d_model))
        pe[:, 0::2] = np.sin(angle[:, 0::2])
        pe[:, 1::2] = np.cos(angle[:, 1::2])
        self.pe = tf.constant(pe, dtype=tf.float32)

    def call(self, x):
        return x + self.pe[: tf.shape(x)[1]]


class TransformerEncoder(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ff = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(d_model)]
        )
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.do1 = layers.Dropout(dropout)
        self.do2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        att = self.attn(x, x)
        att = self.do1(att, training=training)
        out1 = self.ln1(x + att)
        ff = self.ff(out1)
        ff = self.do2(ff, training=training)
        return self.ln2(out1 + ff)


class ReGENTADBackbone(Model):
    def __init__(
        self,
        past_len,
        horizon,
        n_features,
        d_model=128,
        num_heads=6,
        ff_dim=128,
        lstm_units=32,
        dropout=0.1,
        loss_w_y1=0.2,
        loss_w_y2=0.8,
        loss_w_recon=0.5,
        latent_l2=0.0,
    ):
        super().__init__()
        self.past_len = int(past_len)
        self.horizon = int(horizon)
        self.n_features = int(n_features)

        self.loss_w_y1 = float(loss_w_y1)
        self.loss_w_y2 = float(loss_w_y2)
        self.loss_w_recon = float(loss_w_recon)
        self.latent_l2 = float(latent_l2)

        self.ln_in = layers.LayerNormalization(epsilon=1e-6)
        self.conv1 = layers.Conv1D(64, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv1D(64, 3, padding="same", activation="relu")
        self.to_d = layers.Dense(d_model)

        self.pe = PositionalEncoding(self.past_len, d_model)
        self.trans = TransformerEncoder(d_model, num_heads, ff_dim, dropout=dropout)
        self.pool_t = layers.GlobalAveragePooling1D()

        self.lstm = layers.Bidirectional(
            layers.LSTM(lstm_units, return_sequences=True, dropout=dropout)
        )
        self.pool_l = layers.GlobalAveragePooling1D()

        self.concat = layers.Concatenate()
        self.dense_z = layers.Dense(ff_dim, activation="relu")
        self.z_drop = layers.Dropout(dropout)

        self.head_y1 = layers.Dense(self.horizon * self.n_features)
        self.head_recon = layers.Dense(self.past_len * self.n_features)

        self.flat_res = layers.Flatten()
        self.dense_refine = layers.Dense(ff_dim, activation="relu")
        self.head_y2 = layers.Dense(self.horizon * self.n_features)

        self.reshape_y = layers.Reshape((self.horizon, self.n_features))
        self.reshape_x = layers.Reshape((self.past_len, self.n_features))

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def set_loss_weights(self, w_y1, w_y2, w_recon, latent_l2=None):
        self.loss_w_y1 = float(w_y1)
        self.loss_w_y2 = float(w_y2)
        self.loss_w_recon = float(w_recon)
        if latent_l2 is not None:
            self.latent_l2 = float(latent_l2)

    def encode(self, x, training=False):
        x = self.ln_in(x)
        x = self.conv1(x)
        x = self.conv2(x)
        xd = self.to_d(x)

        xt = self.pe(xd)
        xt = self.trans(xt, training=training)
        ht = self.pool_t(xt)

        xl = self.lstm(xd, training=training)
        hl = self.pool_l(xl)

        z = self.concat([ht, hl])
        z = self.dense_z(z)
        z = self.z_drop(z, training=training)

        if self.latent_l2 > 0:
            self.add_loss(self.latent_l2 * tf.reduce_mean(tf.square(z)))
        return z

    def call(self, x, training=False):
        z = self.encode(x, training=training)
        y1 = self.reshape_y(self.head_y1(z))
        rec = self.reshape_x(self.head_recon(z))
        return y1, rec, z

    def refine(self, z, residual):
        r = self.flat_res(residual)
        h = self.dense_refine(tf.concat([z, r], axis=-1))
        return self.reshape_y(self.head_y2(h))

    def train_step(self, data):
        x, targets = data
        y_true = targets["target"]
        x_true = targets["recon"]

        with tf.GradientTape() as tape:
            y1, rec, z = self(x, training=True)
            y2 = self.refine(z, y_true - y1)

            l1 = tf.reduce_mean(tf.square(y_true - y1))
            l2 = tf.reduce_mean(tf.square(y_true - y2))
            lr = tf.reduce_mean(tf.square(x_true - rec))

            loss = self.loss_w_y1 * l1 + self.loss_w_y2 * l2 + self.loss_w_recon * lr
            if self.losses:
                loss += tf.add_n(self.losses)

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}


# ============================================================
# 2. SHARED UTILS
# ============================================================

def _ewma(scores, span):
    if span is None or span <= 1:
        return np.asarray(scores)
    return pd.Series(scores).ewm(span=int(span), adjust=False).mean().values


def _run_length_filter(mask, min_len):
    mask = np.asarray(mask).astype(bool)
    if min_len <= 1:
        return mask.astype(int)

    out = np.zeros_like(mask, dtype=int)
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            if (j - i) >= min_len:
                out[i:j] = 1
            i = j
        else:
            i += 1
    return out


def _dilate_mask(mask, k):
    mask = np.asarray(mask).astype(bool)
    if k <= 0 or mask.size == 0:
        return mask
    out = mask.copy()
    idx = np.where(mask)[0]
    for i in idx:
        lo = max(0, i - k)
        hi = min(len(mask), i + k + 1)
        out[lo:hi] = True
    return out


def _iqr_fit(v):
    q25, q50, q75 = np.quantile(v, [0.25, 0.5, 0.75])
    iqr = max(float(q75 - q25), 1e-6)
    return float(q50), float(iqr)


def _iqr_norm(v, med, iqr):
    iqr = max(float(iqr), 1e-6)
    return np.abs((v - med) / iqr)


def _latent_residual_dynamics(Z, dyn_lag):
    out = np.zeros(len(Z), dtype=float)
    L = int(dyn_lag)
    if len(Z) <= L:
        return out
    for t in range(L, len(Z)):
        out[t] = np.sum((Z[t] - Z[t - L:t].mean(axis=0)) ** 2)
    out[:L] = out[L]
    return out


def _regime_score(Z, z_mu, z_inv_cov):
    if z_mu is None or z_inv_cov is None:
        return np.zeros(len(Z), dtype=float)
    delta = Z - z_mu
    return np.sqrt(np.einsum("nj,jk,nk->n", delta, z_inv_cov, delta))


class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def save_checkpoint_atomic(results, checkpoint_file):
    tmp = checkpoint_file + ".tmp"
    pd.DataFrame(results).to_csv(tmp, index=False)
    os.replace(tmp, checkpoint_file)


def eval_metrics(y_true, y_pred, scores):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    scores = np.asarray(scores, dtype=float)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fpr = ((y_pred == 1) & (y_true == 0)).sum() / max(1, (y_true == 0).sum())

    try:
        au = roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        au = np.nan

    return float(p), float(r), float(f1), float(au), float(fpr)


# ============================================================
# 3. BASE VARIANT CLASS
# ============================================================

class BaseReGENTADVariant:
    def __init__(
        self,
        past_len,
        horizon,
        n_features,
        d_model=128,
        num_heads=6,
        ff_dim=128,
        lstm_units=32,
        dropout=0.1,
        lr=1e-3,
        loss_w_y1=0.2,
        loss_w_y2=0.8,
        loss_w_recon=0.5,
        latent_l2=0.0,
        alpha=0.05,
        n_neighbors=20,
        dyn_lag=5,
        smooth_span=5,
        min_duration=1,
        random_state=42,
    ):
        self.past_len = int(past_len)
        self.horizon = int(horizon)
        self.n_features = int(n_features)

        self.d_model = int(d_model)
        self.num_heads = int(num_heads)
        self.ff_dim = int(ff_dim)
        self.lstm_units = int(lstm_units)
        self.dropout = float(dropout)
        self.lr = float(lr)

        self.loss_w_y1 = float(loss_w_y1)
        self.loss_w_y2 = float(loss_w_y2)
        self.loss_w_recon = float(loss_w_recon)
        self.latent_l2 = float(latent_l2)

        self.alpha = float(alpha)
        self.n_neighbors = int(n_neighbors)
        self.dyn_lag = int(dyn_lag)
        self.smooth_span = int(smooth_span)
        self.min_duration = int(min_duration)
        self.random_state = int(random_state)

        self.network = None
        self.nn = None
        self.stats = {}
        self.threshold_ = None
        self._z_mu = None
        self._z_inv_cov = None
        self._baseline_mu = None
        self._baseline_std = None

    def _build_network(self):
        net = ReGENTADBackbone(
            past_len=self.past_len,
            horizon=self.horizon,
            n_features=self.n_features,
            d_model=self.d_model,
            num_heads=self.num_heads,
            ff_dim=self.ff_dim,
            lstm_units=self.lstm_units,
            dropout=self.dropout,
            loss_w_y1=self.loss_w_y1,
            loss_w_y2=self.loss_w_y2,
            loss_w_recon=self.loss_w_recon,
            latent_l2=self.latent_l2,
        )
        net.compile(optimizer=optimizers.Adam(self.lr))
        return net

    def _purify_indices(self, Xtr, Ytr, q=0.97, max_remove=0.30, epochs=20, batch_size=64, verbose=0):
        tf.keras.backend.clear_session()
        net = self._build_network()
        net.set_loss_weights(w_y1=0.0, w_y2=0.0, w_recon=1.0, latent_l2=self.latent_l2)

        net.fit(
            Xtr,
            {"target": Ytr, "recon": Xtr},
            epochs=int(epochs),
            batch_size=int(batch_size),
            verbose=verbose,
            shuffle=True,
        )

        _, Xrec, _ = net.predict(Xtr, batch_size=256, verbose=0)
        recon_err = np.mean((Xtr - Xrec) ** 2, axis=(1, 2))

        thr = float(np.quantile(recon_err, q))
        cand = recon_err >= thr

        if max_remove is not None:
            max_k = int(np.floor(max_remove * len(Xtr)))
            if max_k < cand.sum():
                idx = np.argsort(recon_err)[::-1][:max_k]
                cand = np.zeros_like(cand, dtype=bool)
                cand[idx] = True

        return np.where(~cand)[0]

    def _fit_backbone(
        self,
        X,
        Y,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        verbose=0,
        purify=True,
        purify_q=0.97,
        purify_max_remove=0.30,
        purify_epochs=20,
        purify_iters=1,
    ):
        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y, dtype=np.float32)

        n = len(X)
        split = int(n * (1 - float(validation_split)))
        Xtr, Xcal = X[:split], X[split:]
        Ytr, Ycal = Y[:split], Y[split:]

        if purify:
            keep = np.arange(len(Xtr))
            for _ in range(int(purify_iters)):
                if len(keep) < max(50, self.n_neighbors + 5):
                    break
                keep2 = self._purify_indices(
                    Xtr[keep],
                    Ytr[keep],
                    q=float(purify_q),
                    max_remove=float(purify_max_remove) if purify_max_remove is not None else None,
                    epochs=int(purify_epochs),
                    batch_size=max(32, int(batch_size)),
                    verbose=verbose,
                )
                keep = keep[keep2]
            Xtr2, Ytr2 = Xtr[keep], Ytr[keep]
        else:
            Xtr2, Ytr2 = Xtr, Ytr

        tf.keras.backend.clear_session()
        self.network = self._build_network()
        self.network.set_loss_weights(
            w_y1=self.loss_w_y1,
            w_y2=self.loss_w_y2,
            w_recon=self.loss_w_recon,
            latent_l2=self.latent_l2,
        )

        self.network.fit(
            Xtr2,
            {"target": Ytr2, "recon": Xtr2},
            epochs=int(epochs),
            batch_size=int(batch_size),
            verbose=verbose,
            shuffle=True,
        )

        self._baseline_mu = Xtr2.mean(axis=(0, 1))
        self._baseline_std = Xtr2.std(axis=(0, 1)) + 1e-8

        return Xtr2, Ytr2, Xcal, Ycal

    def _predict_backbone_outputs(self, X, Y):
        y1, xr, Z = self.network.predict(X, batch_size=256, verbose=0)
        y2 = self.network.refine(Z, Y - y1).numpy()
        resid = Y - y2
        return y1, xr, y2, Z, resid

    def decision_function(self, X, Y):
        raise NotImplementedError

    def predict(self, X, Y, return_scores=False):
        raise NotImplementedError


# ============================================================
# 4. FULL MODEL
# ============================================================

class ReGENTADFull(BaseReGENTADVariant):
    def __init__(
        self,
        past_len,
        horizon,
        n_features,
        weights=None,
        ndt_mode="static",
        window_size=200,
        lag=10,
        sensitivity=1.25,
        enable_rank_mode=True,
        rank_top_frac=0.052,
        rank_min_duration=1,
        rank_dilate=0,
        flag_persistent_regime=False,
        regime_quantile=0.995,
        regime_confirm_len=5,
        regime_persist_max=50,
        **kwargs,
    ):
        super().__init__(past_len, horizon, n_features, **kwargs)

        if weights is None:
            weights = {
                "err": 0.6,
                "recon": 1.2,
                "knn": 0.2,
                "dyn": 0.2,
                "regime": 0.7,
                "vol": 0.6,
            }
        self.weights = dict(weights)

        self.ndt_mode = str(ndt_mode)
        self.window_size = int(window_size)
        self.lag = int(lag)
        self.sensitivity = float(sensitivity)

        self.enable_rank_mode = bool(enable_rank_mode)
        self.rank_top_frac = rank_top_frac
        self.rank_min_duration = int(rank_min_duration)
        self.rank_dilate = int(rank_dilate)

        self.flag_persistent_regime = bool(flag_persistent_regime)
        self.regime_quantile = float(regime_quantile)
        self.regime_confirm_len = int(regime_confirm_len)
        self.regime_persist_max = None if regime_persist_max is None else int(regime_persist_max)

    def fit(self, X, Y, **kwargs):
        Xtr2, Ytr2, Xcal, Ycal = self._fit_backbone(X, Y, purify=True, **kwargs)

        y1_c, xr_c, y2_c, Zcal, resid = self._predict_backbone_outputs(Xcal, Ycal)

        self._z_mu = Zcal.mean(axis=0)
        try:
            cov = np.cov(Zcal.T) + 1e-5 * np.eye(Zcal.shape[1])
            self._z_inv_cov = np.linalg.pinv(cov)
        except Exception:
            var = Zcal.var(axis=0) + 1e-6
            self._z_inv_cov = np.diag(1.0 / var)

        Z_knn = Zcal.copy()
        finite_mask = np.isfinite(Z_knn).all(axis=1)
        if finite_mask.sum() < max(5, self.n_neighbors):
            self.nn = None
        else:
            self.nn = NearestNeighbors(
                n_neighbors=min(self.n_neighbors, finite_mask.sum())
            ).fit(Z_knn[finite_mask])

        if self.nn is not None:
            knn_score = np.log1p(self.nn.kneighbors(Zcal)[0].mean(axis=1))
        else:
            knn_score = np.zeros(len(Zcal))

        raw = {
            "err": np.log1p(np.mean((Ycal - y2_c) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((Xcal - xr_c) ** 2, axis=(1, 2))),
            "knn": knn_score,
            "dyn": np.log1p(_latent_residual_dynamics(Zcal, self.dyn_lag)),
            "regime": np.log1p(_regime_score(Zcal, self._z_mu, self._z_inv_cov)),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        self.stats = {}
        parts = {}
        for k, v in raw.items():
            med, iqr = _iqr_fit(v)
            self.stats[k] = {"med": med, "iqr": iqr}
            parts[k] = _iqr_norm(v, med, iqr) * self.weights.get(k, 1.0)

        scores = np.mean(list(parts.values()), axis=0)
        scores = _ewma(scores, self.smooth_span)

        self.threshold_ = float(np.quantile(scores, 1.0 - self.alpha))
        self.z_threshold = float((self.threshold_ - scores.mean()) / (scores.std() + 1e-8))
        return self

    def decision_function(self, X, Y, return_parts=False):
        y1, xr, y2, Z, resid = self._predict_backbone_outputs(X, Y)

        if self.nn is not None:
            knn_score = np.log1p(self.nn.kneighbors(Z)[0].mean(axis=1))
        else:
            knn_score = np.zeros(len(Z))

        raw = {
            "err": np.log1p(np.mean((Y - y2) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((X - xr) ** 2, axis=(1, 2))),
            "knn": knn_score,
            "dyn": np.log1p(_latent_residual_dynamics(Z, self.dyn_lag)),
            "regime": np.log1p(_regime_score(Z, self._z_mu, self._z_inv_cov)),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        parts = {}
        for k, v in raw.items():
            med = self.stats[k]["med"]
            iqr = self.stats[k]["iqr"]
            parts[k] = _iqr_norm(v, med, iqr) * self.weights.get(k, 1.0)

        scores = np.mean(list(parts.values()), axis=0)
        scores = _ewma(scores, self.smooth_span)

        return (scores, parts) if return_parts else scores

    def _threshold_series(self, scores, adaptive=True):
        sens = max(self.sensitivity, 1e-6)
        thr_static = np.full_like(scores, self.threshold_ / sens, dtype=float)

        if (not adaptive) or (self.ndt_mode == "static"):
            return thr_static

        s = pd.Series(scores).shift(self.lag)
        mu = s.rolling(self.window_size, min_periods=max(20, self.window_size // 5)).mean().bfill()
        sd = s.rolling(self.window_size, min_periods=max(20, self.window_size // 5)).std().bfill().fillna(1e-8)

        zref = max(float(self.z_threshold), 1.0)
        thr_roll = (mu + (zref / sens) * sd).values

        if self.ndt_mode == "adaptive":
            return thr_roll
        if self.ndt_mode == "hybrid":
            return np.maximum(thr_static, thr_roll)
        return thr_static

    def _rank_predict(self, scores):
        N = len(scores)
        if isinstance(self.rank_top_frac, str) and self.rank_top_frac.lower() == "auto":
            top_frac = float(self.alpha)
        else:
            top_frac = float(self.rank_top_frac)

        top_frac = float(np.clip(top_frac, 1e-6, 0.999999))
        k = max(1, min(N, int(np.ceil(top_frac * N))))

        idx = np.argsort(scores)[-k:]
        mask = np.zeros(N, dtype=bool)
        mask[idx] = True
        mask = _dilate_mask(mask, self.rank_dilate)

        return _run_length_filter(mask, self.rank_min_duration)

    def predict(self, X, Y, adaptive=True, return_scores=False):
        scores, parts = self.decision_function(X, Y, return_parts=True)

        if self.enable_rank_mode:
            pred = self._rank_predict(scores)
        else:
            thr = self._threshold_series(scores, adaptive=adaptive)
            pred = _run_length_filter(scores > thr, self.min_duration)

        if self.flag_persistent_regime:
            r = parts["regime"]
            rthr = float(np.quantile(r, self.regime_quantile))
            rmask = r > rthr

            start = None
            cnt = 0
            for i in range(len(rmask)):
                if rmask[i]:
                    cnt += 1
                    if cnt >= self.regime_confirm_len:
                        start = i - self.regime_confirm_len + 1
                        break
                else:
                    cnt = 0

            if start is not None:
                end = len(pred) if self.regime_persist_max is None else min(len(pred), start + self.regime_persist_max)
                pred[start:end] = 1

        if return_scores:
            return pred, scores
        return pred


# ============================================================
# 5. NO DENOISE
# ============================================================
# ============================================================
# 5. NO DENOISE (AGGRESSIVE VERSION)
# ============================================================

class ReGENTADNoDenoise(BaseReGENTADVariant):
    """
    More aggressive no-denoise ablation.

    Differences from full model:
      1. No purification / trimming of suspicious training windows
      2. No robust IQR-based calibration
      3. No calibration on a held-out clean-ish validation split
      4. Threshold is learned directly from contaminated training scores
         using mean/std scaling

    This keeps the same backbone and broad score family, but removes the
    cleaned-reference philosophy that defines the full ReGEN-TAD pipeline.
    """

    def __init__(
        self,
        past_len,
        horizon,
        n_features,
        weights=None,
        rank_top_frac=0.052,
        rank_min_duration=1,
        rank_dilate=0,
        **kwargs,
    ):
        super().__init__(past_len, horizon, n_features, **kwargs)

        if weights is None:
            weights = {
                "err": 0.6,
                "recon": 1.2,
                "knn": 0.2,
                "dyn": 0.2,
                "regime": 0.7,
                "vol": 0.6,
            }
        self.weights = dict(weights)

        self.rank_top_frac = float(rank_top_frac)
        self.rank_min_duration = int(rank_min_duration)
        self.rank_dilate = int(rank_dilate)

        # mean/std score scaling instead of robust median/IQR
        self.score_mu_ = {}
        self.score_sd_ = {}

    def _zscore_fit(self, v):
        mu = float(np.mean(v))
        sd = float(np.std(v))
        sd = max(sd, 1e-6)
        return mu, sd

    def _zscore_norm(self, v, mu, sd):
        sd = max(float(sd), 1e-6)
        return np.abs((v - mu) / sd)

    def _rank_predict(self, scores):
        N = len(scores)
        if N == 0:
            return np.zeros(0, dtype=int)

        top_frac = float(np.clip(self.rank_top_frac, 1e-6, 0.999999))
        k = int(np.ceil(top_frac * N))
        k = max(1, min(N, k))

        idx = np.argsort(scores)[-k:]
        mask = np.zeros(N, dtype=bool)
        mask[idx] = True
        mask = _dilate_mask(mask, self.rank_dilate)
        return _run_length_filter(mask, self.rank_min_duration)

    def fit(
        self,
        X,
        Y,
        epochs=50,
        batch_size=32,
        verbose=0,
        **kwargs,
    ):
        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y, dtype=np.float32)

        # --------------------------------------------------
        # Train directly on all provided windows: no purify,
        # no validation split, no cleaned reference subset
        # --------------------------------------------------
        tf.keras.backend.clear_session()
        self.network = self._build_network()
        self.network.set_loss_weights(
            w_y1=self.loss_w_y1,
            w_y2=self.loss_w_y2,
            w_recon=self.loss_w_recon,
            latent_l2=self.latent_l2,
        )

        self.network.fit(
            X,
            {"target": Y, "recon": X},
            epochs=int(epochs),
            batch_size=int(batch_size),
            verbose=verbose,
            shuffle=True,
        )

        self._baseline_mu = X.mean(axis=(0, 1))
        self._baseline_std = X.std(axis=(0, 1)) + 1e-8

        # --------------------------------------------------
        # Calibrate directly from the raw training scores
        # --------------------------------------------------
        y1_t, xr_t, y2_t, Ztr, resid = self._predict_backbone_outputs(X, Y)

        self._z_mu = Ztr.mean(axis=0)
        try:
            cov = np.cov(Ztr.T) + 1e-5 * np.eye(Ztr.shape[1])
            self._z_inv_cov = np.linalg.pinv(cov)
        except Exception:
            var = Ztr.var(axis=0) + 1e-6
            self._z_inv_cov = np.diag(1.0 / var)

        finite_mask = np.isfinite(Ztr).all(axis=1)
        if finite_mask.sum() < max(5, self.n_neighbors):
            self.nn = None
        else:
            self.nn = NearestNeighbors(
                n_neighbors=min(self.n_neighbors, finite_mask.sum())
            ).fit(Ztr[finite_mask])

        if self.nn is not None:
            knn_score = np.log1p(self.nn.kneighbors(Ztr)[0].mean(axis=1))
        else:
            knn_score = np.zeros(len(Ztr))

        raw = {
            "err": np.log1p(np.mean((Y - y2_t) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((X - xr_t) ** 2, axis=(1, 2))),
            "knn": knn_score,
            "dyn": np.log1p(_latent_residual_dynamics(Ztr, self.dyn_lag)),
            "regime": np.log1p(_regime_score(Ztr, self._z_mu, self._z_inv_cov)),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        self.score_mu_ = {}
        self.score_sd_ = {}
        parts = {}

        for k, v in raw.items():
            mu, sd = self._zscore_fit(v)
            self.score_mu_[k] = mu
            self.score_sd_[k] = sd
            parts[k] = self._zscore_norm(v, mu, sd) * self.weights.get(k, 1.0)

        scores = np.mean(list(parts.values()), axis=0)
        scores = _ewma(scores, self.smooth_span)

        # threshold directly from contaminated training scores
        self.threshold_ = float(np.quantile(scores, 1.0 - self.alpha))
        return self

    def decision_function(self, X, Y):
        if self.network is None:
            raise RuntimeError("Call fit() before decision_function().")

        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y, dtype=np.float32)

        y1, xr, y2, Z, resid = self._predict_backbone_outputs(X, Y)

        if self.nn is not None:
            knn_score = np.log1p(self.nn.kneighbors(Z)[0].mean(axis=1))
        else:
            knn_score = np.zeros(len(Z))

        raw = {
            "err": np.log1p(np.mean((Y - y2) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((X - xr) ** 2, axis=(1, 2))),
            "knn": knn_score,
            "dyn": np.log1p(_latent_residual_dynamics(Z, self.dyn_lag)),
            "regime": np.log1p(_regime_score(Z, self._z_mu, self._z_inv_cov)),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        parts = {}
        for k, v in raw.items():
            mu = self.score_mu_[k]
            sd = self.score_sd_[k]
            parts[k] = self._zscore_norm(v, mu, sd) * self.weights.get(k, 1.0)

        scores = np.mean(list(parts.values()), axis=0)
        scores = _ewma(scores, self.smooth_span)
        return scores

    def predict(self, X, Y, return_scores=False):
        scores = self.decision_function(X, Y)

        # still use rank-based decision for comparability,
        # but thresholding/calibration is no longer robust/cleaned
        pred = self._rank_predict(scores)

        if return_scores:
            return pred, scores
        return pred


# ============================================================
# 6. BACKBONE ONLY
# ============================================================

class ReGENTADBackboneOnly(BaseReGENTADVariant):
    """
    Uses the same backbone but relies on reconstruction score only.
    This is intentionally more different from the full method.
    """
    def fit(self, X, Y, **kwargs):
        _, _, Xcal, Ycal = self._fit_backbone(X, Y, purify=True, **kwargs)
        _, xr_c, _, _, _ = self._predict_backbone_outputs(Xcal, Ycal)

        recon_score = np.log1p(np.mean((Xcal - xr_c) ** 2, axis=(1, 2)))
        med, iqr = _iqr_fit(recon_score)

        self.stats = {"recon": {"med": med, "iqr": iqr}}
        scores = _iqr_norm(recon_score, med, iqr)
        scores = _ewma(scores, self.smooth_span)

        self.threshold_ = float(np.quantile(scores, 1.0 - self.alpha))
        return self

    def decision_function(self, X, Y):
        _, xr, _, _, _ = self._predict_backbone_outputs(X, Y)
        recon_score = np.log1p(np.mean((X - xr) ** 2, axis=(1, 2)))
        med = self.stats["recon"]["med"]
        iqr = self.stats["recon"]["iqr"]
        scores = _iqr_norm(recon_score, med, iqr)
        return _ewma(scores, self.smooth_span)

    def predict(self, X, Y, return_scores=False):
        scores = self.decision_function(X, Y)
        pred = _run_length_filter(scores > self.threshold_, self.min_duration)
        if return_scores:
            return pred, scores
        return pred


# ============================================================
# 7. NO CALIBRATION
# ============================================================

class ReGENTADNoCalibration(BaseReGENTADVariant):
    """
    Uses multiple components, but removes robust validation-based calibration.
    Threshold is learned naively from training-region scores.
    """
    def fit(self, X, Y, **kwargs):
        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y, dtype=np.float32)

        tf.keras.backend.clear_session()
        self.network = self._build_network()
        self.network.set_loss_weights(
            w_y1=self.loss_w_y1,
            w_y2=self.loss_w_y2,
            w_recon=self.loss_w_recon,
            latent_l2=self.latent_l2,
        )

        self.network.fit(
            X,
            {"target": Y, "recon": X},
            epochs=int(kwargs.get("epochs", 50)),
            batch_size=int(kwargs.get("batch_size", 32)),
            verbose=int(kwargs.get("verbose", 0)),
            shuffle=True,
        )

        self._baseline_mu = X.mean(axis=(0, 1))
        self._baseline_std = X.std(axis=(0, 1)) + 1e-8

        y1, xr, y2, Z, resid = self._predict_backbone_outputs(X, Y)

        self._z_mu = Z.mean(axis=0)
        cov = np.cov(Z.T) + 1e-5 * np.eye(Z.shape[1])
        self._z_inv_cov = np.linalg.pinv(cov)

        finite_mask = np.isfinite(Z).all(axis=1)
        if finite_mask.sum() >= max(5, self.n_neighbors):
            self.nn = NearestNeighbors(
                n_neighbors=min(self.n_neighbors, finite_mask.sum())
            ).fit(Z[finite_mask])
        else:
            self.nn = None

        if self.nn is not None:
            knn_score = np.log1p(self.nn.kneighbors(Z)[0].mean(axis=1))
        else:
            knn_score = np.zeros(len(Z))

        raw = {
            "err": np.log1p(np.mean((Y - y2) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((X - xr) ** 2, axis=(1, 2))),
            "knn": knn_score,
            "dyn": np.log1p(_latent_residual_dynamics(Z, self.dyn_lag)),
            "regime": np.log1p(_regime_score(Z, self._z_mu, self._z_inv_cov)),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        # Naive mean-of-raw-scores, no robust scaling, no validation calibration
        scores = np.mean(list(raw.values()), axis=0)
        self.threshold_ = float(scores.mean() + 3.0 * (scores.std() + 1e-8))
        return self

    def decision_function(self, X, Y):
        y1, xr, y2, Z, resid = self._predict_backbone_outputs(X, Y)

        if self.nn is not None:
            knn_score = np.log1p(self.nn.kneighbors(Z)[0].mean(axis=1))
        else:
            knn_score = np.zeros(len(Z))

        raw = {
            "err": np.log1p(np.mean((Y - y2) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((X - xr) ** 2, axis=(1, 2))),
            "knn": knn_score,
            "dyn": np.log1p(_latent_residual_dynamics(Z, self.dyn_lag)),
            "regime": np.log1p(_regime_score(Z, self._z_mu, self._z_inv_cov)),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        return np.mean(list(raw.values()), axis=0)

    def predict(self, X, Y, return_scores=False):
        scores = self.decision_function(X, Y)
        pred = _run_length_filter(scores > self.threshold_, self.min_duration)
        if return_scores:
            return pred, scores
        return pred


# ============================================================
# 8. NO LATENT
# ============================================================

class ReGENTADNoLatent(BaseReGENTADVariant):
    """
    Removes latent-space diagnostics: knn, dyn, regime.
    Keeps err, recon, vol with robust calibration.
    """
    def fit(self, X, Y, **kwargs):
        _, _, Xcal, Ycal = self._fit_backbone(X, Y, purify=True, **kwargs)

        y1_c, xr_c, y2_c, _, resid = self._predict_backbone_outputs(Xcal, Ycal)

        raw = {
            "err": np.log1p(np.mean((Ycal - y2_c) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((Xcal - xr_c) ** 2, axis=(1, 2))),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        weights = {"err": 0.6, "recon": 1.2, "vol": 0.6}

        self.stats = {}
        parts = {}
        for k, v in raw.items():
            med, iqr = _iqr_fit(v)
            self.stats[k] = {"med": med, "iqr": iqr}
            parts[k] = _iqr_norm(v, med, iqr) * weights[k]

        scores = np.mean(list(parts.values()), axis=0)
        scores = _ewma(scores, self.smooth_span)

        self.threshold_ = float(np.quantile(scores, 1.0 - self.alpha))
        self.rank_top_frac = 0.052
        return self

    def decision_function(self, X, Y):
        y1, xr, y2, _, resid = self._predict_backbone_outputs(X, Y)

        raw = {
            "err": np.log1p(np.mean((Y - y2) ** 2, axis=(1, 2))),
            "recon": np.log1p(np.mean((X - xr) ** 2, axis=(1, 2))),
            "vol": np.log1p(np.std(resid, axis=(1, 2))),
        }

        weights = {"err": 0.6, "recon": 1.2, "vol": 0.6}
        parts = {}
        for k, v in raw.items():
            med = self.stats[k]["med"]
            iqr = self.stats[k]["iqr"]
            parts[k] = _iqr_norm(v, med, iqr) * weights[k]

        scores = np.mean(list(parts.values()), axis=0)
        return _ewma(scores, self.smooth_span)

    def predict(self, X, Y, return_scores=False):
        scores = self.decision_function(X, Y)

        N = len(scores)
        k = max(1, min(N, int(np.ceil(0.052 * N))))
        idx = np.argsort(scores)[-k:]
        mask = np.zeros(N, dtype=bool)
        mask[idx] = True
        pred = _run_length_filter(mask, self.min_duration)

        if return_scores:
            return pred, scores
        return pred


# ============================================================
# 9. SYNTHETIC DATA FOR ABLATION
# ============================================================

def generate_stock_like_data(
    n_normal=450,
    n_shock=50,
    p=100,
    anomaly_type="mean_shift",
    seed=0,
):
    rng = np.random.default_rng(seed)

    mu0 = rng.uniform(-0.0005, 0.0005)
    sig0 = rng.uniform(0.007, 0.015)

    R0 = rng.normal(mu0, sig0, size=(n_normal, p))
    affected = np.arange(p)
    R1 = np.zeros((n_shock, p))

    if anomaly_type == "mean_shift":
        mu = rng.uniform(0.01, 0.04)
        market = rng.normal(mu, sig0, size=(n_shock, 1))
        idio = rng.normal(0, sig0 * 0.5, size=(n_shock, p))
        R1[:, affected] = market + idio

    elif anomaly_type == "variance":
        sig = rng.uniform(0.03, 0.06)
        R1[:, affected] = rng.normal(mu0, sig, size=(n_shock, p))

    elif anomaly_type == "trend":
        mu_trend = rng.uniform(0.01, 0.04)
        trend = np.linspace(0, mu_trend, n_shock)[:, None]
        R1[:, affected] = rng.normal(mu0, sig0, size=(n_shock, p)) + trend

    elif anomaly_type == "spike":
        R1[:] = rng.normal(mu0, sig0, size=(n_shock, p))
        spike_t = rng.integers(0, n_shock)
        R1[spike_t, affected] += rng.choice([-1, 1]) * rng.uniform(0.10, 0.25)

    elif anomaly_type == "collective":
        collective_level = rng.uniform(0.01, 0.04)
        collective_noise = rng.normal(0, sig0 * 0.15, size=(1, p))
        R1[:, affected] = collective_level + collective_noise

    elif anomaly_type == "contextual":
        base = rng.normal(mu0, sig0, size=(n_shock, p))
        context_boost = rng.uniform(0.02, 0.05)
        R1[:, affected] = base + context_boost * (base > 0)

    else:
        raise ValueError(f"Unknown anomaly_type: {anomaly_type}")

    X = np.vstack([R0, R1])
    y = np.zeros(len(X), dtype=int)
    y[n_normal:] = 1
    return X, y


def make_windows(X, y, past_len, horizon):
    Xp, Yf, yw = [], [], []
    for t in range(past_len, len(X) - horizon):
        Xp.append(X[t - past_len:t])
        Yf.append(X[t:t + horizon])
        yw.append(int(y[t:t + horizon].max()))
    return np.asarray(Xp), np.asarray(Yf), np.asarray(yw)


def contaminate_training_data(Xp_tr, Yf_tr, contam_rate, rng):
    if contam_rate <= 0:
        return Xp_tr, Yf_tr

    n_train = len(Xp_tr)
    n_contam = int(np.ceil(contam_rate * n_train))

    Xp_contam = Xp_tr.copy()
    Yf_contam = Yf_tr.copy()

    # mix of isolated and clustered contamination
    contam_mode = rng.choice(
        ["isolated", "clustered", "persistent_drift", "heavy_tail", "dependence_break"],
        p=[0.20, 0.25, 0.20, 0.20, 0.15],
    )

    if contam_mode == "isolated":
        contam_idx = rng.choice(n_train, n_contam, replace=False)

    elif contam_mode == "clustered":
        start = rng.integers(0, max(1, n_train - n_contam))
        contam_idx = np.arange(start, min(n_train, start + n_contam))

    elif contam_mode == "persistent_drift":
        start = rng.integers(0, max(1, n_train - n_contam))
        contam_idx = np.arange(start, min(n_train, start + n_contam))

    elif contam_mode == "heavy_tail":
        contam_idx = rng.choice(n_train, n_contam, replace=False)

    elif contam_mode == "dependence_break":
        contam_idx = rng.choice(n_train, n_contam, replace=False)

    for j, idx in enumerate(contam_idx):
        if contam_mode == "isolated":
            contam_type = rng.choice(["shift", "scale", "spike", "noise"])
            if contam_type == "shift":
                shift = rng.uniform(-0.05, 0.05)
                Xp_contam[idx] += shift
                Yf_contam[idx] += shift
            elif contam_type == "scale":
                scale = rng.uniform(1.5, 3.0)
                Xp_contam[idx] *= scale
                Yf_contam[idx] *= scale
            elif contam_type == "spike":
                n_spikes = rng.integers(1, 4)
                spike_t = rng.choice(Xp_contam.shape[1], n_spikes, replace=False)
                spike_f = rng.choice(Xp_contam.shape[2], n_spikes, replace=True)
                for t, f in zip(spike_t, spike_f):
                    Xp_contam[idx, t, f] += rng.choice([-1, 1]) * rng.uniform(0.1, 0.3)
            else:
                noise_scale = rng.uniform(2.0, 4.0)
                Xp_contam[idx] += rng.normal(0, 0.01 * noise_scale, Xp_contam[idx].shape)
                Yf_contam[idx] += rng.normal(0, 0.01 * noise_scale, Yf_contam[idx].shape)

        elif contam_mode == "clustered":
            shift = rng.uniform(-0.03, 0.03)
            scale = rng.uniform(1.4, 2.0)
            Xp_contam[idx] = scale * Xp_contam[idx] + shift
            Yf_contam[idx] = scale * Yf_contam[idx] + shift

        elif contam_mode == "persistent_drift":
            frac = j / max(1, len(contam_idx) - 1)
            drift = rng.uniform(0.01, 0.05) * frac
            Xp_contam[idx] += drift
            Yf_contam[idx] += drift

        elif contam_mode == "heavy_tail":
            Xp_contam[idx] += rng.standard_t(df=rng.uniform(2.5, 4.0), size=Xp_contam[idx].shape) * 0.03
            Yf_contam[idx] += rng.standard_t(df=rng.uniform(2.5, 4.0), size=Yf_contam[idx].shape) * 0.03

        elif contam_mode == "dependence_break":
            # correlated contamination without huge marginal mean change
            common = rng.normal(0, 0.03, size=(Xp_contam.shape[1], 1))
            load = rng.normal(1.0, 0.2, size=(1, Xp_contam.shape[2]))
            Xp_contam[idx] += common @ load
            Yf_contam[idx] += rng.normal(0, 0.02, size=Yf_contam[idx].shape)

    return Xp_contam, Yf_contam

def generate_stock_like_data(
    n_normal=450,
    n_shock=50,
    p=250,
    anomaly_type="bear_market",
    shock_sign="random",
    frac_affected=0.5,
    seed=0,
):
    rng = np.random.default_rng(seed)

    mu0 = rng.uniform(-0.0005, 0.0005)
    sig0 = rng.uniform(0.007, 0.015)
    R0 = rng.normal(mu0, sig0, size=(n_normal, p))

    if shock_sign == "random":
        sign = rng.choice([-1, 1])
    elif shock_sign == "positive":
        sign = 1
    else:
        sign = -1

    n_aff = max(1, int(np.ceil(frac_affected * p)))
    affected = rng.choice(p, n_aff, replace=False)
    unaffected = np.setdiff1d(np.arange(p), affected)
    R1 = np.zeros((n_shock, p))

    if anomaly_type in {"bear_market", "bull_market", "mean_shift"}:
        mu = sign * rng.uniform(0.01, 0.04)
        market = rng.normal(mu, sig0, size=(n_shock, 1))
        idio = rng.normal(0, sig0 * 0.5, size=(n_shock, n_aff))
        R1[:, affected] = market + idio

    elif anomaly_type == "volatility_spike":
        sig = rng.uniform(0.03, 0.06)
        R1[:, affected] = rng.normal(mu0, sig, size=(n_shock, n_aff))

    elif anomaly_type == "trend_reversal":
        mu_trend = -mu0 * rng.uniform(4, 6)
        market = rng.normal(mu_trend, sig0 * 1.5, size=(n_shock, 1))
        R1[:, affected] = market

    elif anomaly_type == "flash_crash":
        R1[:] = rng.normal(mu0, sig0, size=(n_shock, p))
        crash_t = rng.integers(0, n_shock)
        R1[crash_t, affected] -= rng.uniform(0.15, 0.30)

    elif anomaly_type == "sector_shock":
        mu = sign * rng.uniform(0.02, 0.05)
        R1[:, affected] = rng.normal(mu, sig0 * 1.5, size=(n_shock, n_aff))

    elif anomaly_type == "liquidity_dryup":
        R1[:, affected] = rng.normal(mu0, sig0 * 4.0, size=(n_shock, n_aff))

    elif anomaly_type == "regime_switch":
        mu = sign * rng.uniform(0.01, 0.03)
        sig = rng.uniform(0.03, 0.06)
        market = rng.normal(mu, sig, size=(n_shock, 1))
        idio = rng.normal(0, sig, size=(n_shock, n_aff))
        R1[:, affected] = market + idio

    elif anomaly_type == "correlation_breakdown":
        sig_shock = sig0 * rng.uniform(2.0, 3.0)
        R1[:, affected] = rng.normal(0, sig_shock, size=(n_shock, n_aff))
        n_spikes = max(1, n_shock // 10)
        spike_times = rng.choice(n_shock, n_spikes, replace=False)
        spike_assets = rng.choice(n_aff, n_spikes, replace=True)
        for t, a in zip(spike_times, spike_assets):
            R1[t, affected[a]] += rng.choice([-1, 1]) * rng.uniform(0.05, 0.15)

    elif anomaly_type == "contagion":
        n_initial = max(1, n_aff // 5)
        spread_rate = (n_aff - n_initial) / max(1, n_shock - 1)
        mu_shock = sign * rng.uniform(0.02, 0.04)
        sig_shock = sig0 * 1.5
        for t in range(n_shock):
            n_affected_t = min(n_aff, int(n_initial + spread_rate * t))
            affected_t = affected[:n_affected_t]
            R1[t, affected_t] = rng.normal(mu_shock, sig_shock, size=n_affected_t)
            if len(unaffected) > 0:
                R1[t, unaffected] = rng.normal(mu0, sig0, size=len(unaffected))

    elif anomaly_type == "momentum_crash":
        n_winners = n_aff // 2
        winners = affected[:n_winners]
        losers = affected[n_winners:]
        mu_reversal = rng.uniform(0.03, 0.06)
        sig_shock = sig0 * 2.0
        R1[:, winners] = rng.normal(-mu_reversal, sig_shock, size=(n_shock, len(winners)))
        if len(losers) > 0:
            R1[:, losers] = rng.normal(mu_reversal, sig_shock, size=(n_shock, len(losers)))

    elif anomaly_type == "fat_tail_event":
        df_t = rng.uniform(2.5, 4.0)
        scale = sig0 * 1.5
        R1[:, affected] = rng.standard_t(df_t, size=(n_shock, n_aff)) * scale
        n_extreme = max(1, n_shock // 5)
        extreme_times = rng.choice(n_shock, n_extreme, replace=False)
        extreme_assets = rng.choice(n_aff, n_extreme, replace=True)
        for t, a in zip(extreme_times, extreme_assets):
            R1[t, affected[a]] += rng.choice([-1, 1]) * rng.uniform(0.10, 0.25)

    elif anomaly_type == "microstructure_noise":
        base_returns = rng.normal(mu0, sig0, size=(n_shock, n_aff))
        bounce_amplitude = rng.uniform(0.005, 0.015)
        bounce = np.zeros((n_shock, n_aff))
        for i in range(n_aff):
            phase = rng.uniform(0, 2 * np.pi)
            freq = rng.uniform(0.3, 0.7)
            bounce[:, i] = bounce_amplitude * np.sin(freq * np.arange(n_shock) + phase)
        noise_bursts = rng.choice(n_shock, size=max(1, n_shock // 10), replace=False)
        burst_noise = np.zeros((n_shock, n_aff))
        for t in noise_bursts:
            burst_noise[t, :] = rng.normal(0, sig0 * 3, size=n_aff)
        R1[:, affected] = base_returns + bounce + burst_noise

    # -------- NEW ANOMALIES --------

    elif anomaly_type == "low_signal_contagion":
        n_initial = max(1, n_aff // 8)
        spread_rate = (n_aff - n_initial) / max(1, n_shock - 1)
        mu_shock = sign * rng.uniform(0.004, 0.010)
        sig_shock = sig0 * 1.15
        for t in range(n_shock):
            n_affected_t = min(n_aff, int(n_initial + spread_rate * t))
            affected_t = affected[:n_affected_t]
            R1[t, affected_t] = rng.normal(mu_shock, sig_shock, size=n_affected_t)
            if len(unaffected) > 0:
                R1[t, unaffected] = rng.normal(mu0, sig0, size=len(unaffected))

    elif anomaly_type == "latent_cluster_shift":
        # small mean signal, strong shared latent factor
        latent = rng.normal(sign * rng.uniform(0.006, 0.012), sig0 * 0.4, size=(n_shock, 1))
        loadings = rng.normal(1.0, 0.15, size=(1, n_aff))
        idio = rng.normal(0, sig0 * 0.35, size=(n_shock, n_aff))
        R1[:, affected] = latent @ loadings + idio

    elif anomaly_type == "covariance_rotation":
        # means unchanged, variances similar, dependence changes
        base = rng.normal(mu0, sig0, size=(n_shock, n_aff))
        common1 = rng.normal(0, sig0 * 1.8, size=(n_shock, 1))
        common2 = rng.normal(0, sig0 * 1.8, size=(n_shock, 1))
        half = max(1, n_aff // 2)
        R1[:, affected[:half]] = 0.75 * common1 + 0.25 * base[:, :half]
        R1[:, affected[half:]] = -0.75 * common2 + 0.25 * base[:, half:]

    elif anomaly_type == "stochastic_vol_regime":
        vol = np.zeros(n_shock)
        vol[0] = sig0 * 1.2
        for t in range(1, n_shock):
            vol[t] = 0.90 * vol[t-1] + 0.10 * rng.uniform(sig0 * 2.0, sig0 * 4.0)
        shocks = rng.normal(0, 1, size=(n_shock, n_aff))
        R1[:, affected] = mu0 + shocks * vol[:, None]

    elif anomaly_type == "mixed_regime":
        latent = rng.normal(sign * rng.uniform(0.006, 0.015), sig0 * 0.5, size=(n_shock, 1))
        idio = rng.normal(0, sig0 * rng.uniform(1.8, 2.8), size=(n_shock, n_aff))
        drift = np.linspace(0, sign * rng.uniform(0.004, 0.012), n_shock)[:, None]
        R1[:, affected] = latent + drift + idio

    elif anomaly_type == "adversarial_contextual":
        base = rng.normal(mu0, sig0, size=(n_shock, n_aff))
        local_state = np.sign(np.cumsum(rng.normal(0, 1, size=(n_shock, 1))))
        boost = rng.uniform(0.01, 0.025)
        R1[:, affected] = base + boost * (base > 0) * local_state

    else:
        raise ValueError(f"Unknown anomaly_type: {anomaly_type}")

    X = np.vstack([R0, R1])
    y = np.zeros(len(X), dtype=int)
    y[n_normal:] = 1
    return X, y


# ============================================================
# 10. ABLATION RUNNER
# ============================================================

PAST_LEN = 24
HORIZON = 6
N_ITER = 10

DIMENSIONS = [50]
SAMPLE_SIZES = [(450, 50)]
#CONTAMINATION_RATES = [0.15]
CONTAMINATION_RATES = [0.03, 0.10, 0.15, 0.20]
ANOMALIES = ["mean_shift", "variance", "trend", "spike", "collective", "contextual",
    "low_signal_contagion",
    "latent_cluster_shift",
    "covariance_rotation",
    "stochastic_vol_regime",
    "mixed_regime",
    "adversarial_contextual",
]

VARIANT_CLASSES = {
    "A0": ("full", ReGENTADFull),
    "A1": ("no_denoise", ReGENTADNoDenoise),
    "A2": ("backbone_only", ReGENTADBackboneOnly),
    "A3": ("no_calibration", ReGENTADNoCalibration),
    "A4": ("no_latent", ReGENTADNoLatent),
}

CHECKPOINT_FILE = "ablation_checkpoint.csv"
RESULTS_FILE = "ablation_results_full.csv"
SUMMARY_FILE = "ablation_summary.csv"


def run_one_variant(VariantClass, past_len, horizon, dim, Xp_tr_contam, Yf_tr_contam, Xp, Yf):
    model = VariantClass(
        past_len=past_len,
        horizon=horizon,
        n_features=dim,
    )
    model.fit(
        Xp_tr_contam,
        Yf_tr_contam,
        epochs=40,
        batch_size=32,
        verbose=0,
    )
    yhat, scores = model.predict(Xp, Yf, return_scores=True)
    return yhat, scores


def run_ablation_study():
    results = []
    completed = set()

    if os.path.exists(CHECKPOINT_FILE):
        try:
            ckpt = pd.read_csv(CHECKPOINT_FILE)
            results = ckpt.to_dict("records")
            for row in results:
                completed.add((
                    row["Variant"],
                    row["Anomaly"],
                    int(row["Iteration"]),
                    int(row["Dim"]),
                    int(row["N_Normal"]),
                    int(row["N_Shock"]),
                    round(float(row["ContamRate"]), 6),
                ))
            print(f"Resuming from checkpoint: {CHECKPOINT_FILE} ({len(results)} rows)")
        except Exception as e:
            print(f"Checkpoint read failed ({e}); starting fresh.")

    total_jobs = (
        len(DIMENSIONS)
        * len(SAMPLE_SIZES)
        * len(ANOMALIES)
        * len(CONTAMINATION_RATES)
        * N_ITER
        * len(VARIANT_CLASSES)
    )

    done = 0
    t_global = time.perf_counter()

    for dim in DIMENSIONS:
        for (n_normal, n_shock) in SAMPLE_SIZES:
            for anomaly in ANOMALIES:
                for contam_rate in CONTAMINATION_RATES:
                    for it in range(N_ITER):
                        seed = 1000 + it
                        rng = np.random.default_rng(seed)
                        set_all_seeds(seed)

                        try:
                            X, y = generate_stock_like_data(
                                n_normal=n_normal,
                                n_shock=n_shock,
                                p=dim,
                                anomaly_type=anomaly,
                                seed=seed,
                            )

                            Xp, Yf, y_eval = make_windows(X, y, PAST_LEN, HORIZON)

                            shock_positions = np.where(y_eval == 1)[0]
                            if len(shock_positions) == 0:
                                continue

                            shock_start = int(shock_positions[0])
                            Xp_tr = Xp[:shock_start]
                            Yf_tr = Yf[:shock_start]

                            if len(Xp_tr) < 20:
                                continue

                            Xp_tr_contam, Yf_tr_contam = contaminate_training_data(
                                Xp_tr, Yf_tr, contam_rate, rng
                            )

                            base_result = dict(
                                Anomaly=anomaly,
                                Iteration=it,
                                Dim=dim,
                                N_Normal=n_normal,
                                N_Shock=n_shock,
                                ContamRate=contam_rate,
                            )

                            for variant_label, (variant_name, VariantClass) in VARIANT_CLASSES.items():
                                done += 1
                                print(
                                    f"[{done}/{total_jobs}] "
                                    f"{variant_label} | {variant_name} | "
                                    f"anomaly={anomaly} | iter={it} | "
                                    f"elapsed={time.perf_counter()-t_global:.1f}s"
                                )

                                key = (
                                    variant_label,
                                    anomaly,
                                    it,
                                    dim,
                                    n_normal,
                                    n_shock,
                                    round(float(contam_rate), 6),
                                )
                                if key in completed:
                                    continue

                                try:
                                    tf.keras.backend.clear_session()
                                    gc.collect()
                                    set_all_seeds(seed)

                                    with Timer() as t:
                                        yhat, scores = run_one_variant(
                                            VariantClass,
                                            PAST_LEN,
                                            HORIZON,
                                            dim,
                                            Xp_tr_contam,
                                            Yf_tr_contam,
                                            Xp,
                                            Yf,
                                        )

                                    p, r, f1, au, fpr = eval_metrics(y_eval, yhat, scores)

                                    row = {
                                        **base_result,
                                        "Variant": variant_label,
                                        "VariantName": variant_name,
                                        "Precision": p,
                                        "Recall": r,
                                        "F1": f1,
                                        "AUROC": au,
                                        "FPR": fpr,
                                        "Time": t.elapsed,
                                    }

                                    results.append(row)
                                    completed.add(key)
                                    save_checkpoint_atomic(results, CHECKPOINT_FILE)

                                except Exception as e:
                                    print(f"  {variant_label} ERROR: {e}")

                        except Exception as e:
                            print(f"  Scenario ERROR: {e}")

    df = pd.DataFrame(results)
    df.to_csv(RESULTS_FILE, index=False)

    summary = (
        df.groupby(["Variant", "VariantName"], as_index=False)[
            ["Precision", "Recall", "F1", "AUROC", "FPR", "Time"]
        ]
        .mean()
        .sort_values("Variant")
        .reset_index(drop=True)
    )
    summary.to_csv(SUMMARY_FILE, index=False)

    print("\nAblation study complete.")
    print(f"Rows: {len(df)}")
    print(f"Results saved to: {RESULTS_FILE}")
    print(f"Summary saved to: {SUMMARY_FILE}")

    return df, summary


# ============================================================
# 11. RUN
# ============================================================

df_ablation, df_ablation_summary = run_ablation_study()
display(df_ablation_summary.round(4))

---
# Section 2 — Branch-Level Ablation (Expanded, B0–B9)

This section adds a **10-variant branch-level ablation** to the original
pipeline-level study.  It is fully additive: no code from Section 1 is
modified.

### Architecture components being toggled

| Component | Role |
|-----------|------|
| CNN | Two stacked Conv1D layers extracting local temporal patterns |
| Transformer | PositionalEncoding + TransformerEncoder for long-range dependencies |
| BiLSTM | Bidirectional LSTM capturing sequential structure in both directions |
| Refinement | Residual correction head (Y2) trained on top of the direct forecast (Y1) |

### No Refinement design (B4)

`BranchVariantExpanded` sets `loss_w_y2=0.0` during training to prevent the
refinement head from influencing the backbone.  At inference, `y1` (direct
forecast) replaces `y2` (refined forecast) in all score components.


In [ ]:
# ============================================================
# BRANCH-LEVEL EXTENSION — Configurable Backbone & Variant
# ============================================================
# These classes are additive: they do not alter any class
# defined in Section 1.
# ============================================================


class ConfigurableReGENTADBackbone(ReGENTADBackbone):
    """
    Minimal subclass of the shared ReGENTADBackbone that adds three
    toggle flags for the three parallel encoding branches.

    Flags
    -----
    use_cnn         : if False, the two Conv1D layers are skipped;
                      the raw (layer-normed) input is projected to d_model.
    use_transformer : if False, the PositionalEncoding + TransformerEncoder
                      branch is skipped.
    use_bilstm      : if False, the Bidirectional LSTM branch is skipped.

    When both use_transformer and use_bilstm are False a
    GlobalAveragePooling1D is applied on the to_d output as a fallback
    so that a latent vector is always produced.
    """

    def __init__(
        self,
        past_len,
        horizon,
        n_features,
        use_cnn=True,
        use_transformer=True,
        use_bilstm=True,
        **kwargs,
    ):
        super().__init__(past_len, horizon, n_features, **kwargs)
        self._use_cnn         = bool(use_cnn)
        self._use_transformer = bool(use_transformer)
        self._use_bilstm      = bool(use_bilstm)
        if not self._use_transformer and not self._use_bilstm:
            self._pool_direct = layers.GlobalAveragePooling1D()

    def encode(self, x, training=False):
        """
        Conditional encoding: only the active branches contribute to the
        latent vector z.  Uses tf.concat directly to handle 0, 1, or 2
        active branches without triggering the parent Concatenate layer.
        """
        x = self.ln_in(x)
        if self._use_cnn:
            x = self.conv1(x)
            x = self.conv2(x)
        xd = self.to_d(x)

        parts = []
        if self._use_transformer:
            xt = self.pe(xd)
            xt = self.trans(xt, training=training)
            parts.append(self.pool_t(xt))
        if self._use_bilstm:
            xl = self.lstm(xd, training=training)
            parts.append(self.pool_l(xl))

        if len(parts) == 0:
            z_combined = self._pool_direct(xd)
        elif len(parts) == 1:
            z_combined = parts[0]
        else:
            z_combined = tf.concat(parts, axis=-1)

        z = self.dense_z(z_combined)
        z = self.z_drop(z, training=training)
        if self.latent_l2 > 0:
            self.add_loss(self.latent_l2 * tf.reduce_mean(tf.square(z)))
        return z


class BranchVariantExpanded(ReGENTADFull):
    """
    Expanded branch-level ablation variant.

    Adds a use_refinement flag on top of the existing branch toggles.

    When use_refinement=False
    -------------------------
    * loss_w_y2 is set to 0.0 so the refinement head does not affect
      the backbone during training.
    * _predict_backbone_outputs() substitutes y1 for y2, so the
      prediction-error score component ("err") is computed against the
      direct forecast rather than the refined forecast.  All other
      components (recon, knn, dyn, regime, vol) are unaffected.

    Parameters
    ----------
    use_cnn, use_transformer, use_bilstm, use_refinement : bool
    **kwargs : forwarded to ReGENTADFull (and BaseReGENTADVariant)
    """

    def __init__(
        self,
        past_len,
        horizon,
        n_features,
        use_cnn=True,
        use_transformer=True,
        use_bilstm=True,
        use_refinement=True,
        **kwargs,
    ):
        super().__init__(past_len, horizon, n_features, **kwargs)
        self._use_cnn         = bool(use_cnn)
        self._use_transformer = bool(use_transformer)
        self._use_bilstm      = bool(use_bilstm)
        self._use_refinement  = bool(use_refinement)

    def _build_network(self):
        """
        Swap in ConfigurableReGENTADBackbone.
        When use_refinement=False, pass loss_w_y2=0.0 so the refinement
        head is not trained.
        """
        net = ConfigurableReGENTADBackbone(
            past_len=self.past_len,
            horizon=self.horizon,
            n_features=self.n_features,
            use_cnn=self._use_cnn,
            use_transformer=self._use_transformer,
            use_bilstm=self._use_bilstm,
            d_model=self.d_model,
            num_heads=self.num_heads,
            ff_dim=self.ff_dim,
            lstm_units=self.lstm_units,
            dropout=self.dropout,
            loss_w_y1=self.loss_w_y1,
            loss_w_y2=self.loss_w_y2 if self._use_refinement else 0.0,
            loss_w_recon=self.loss_w_recon,
            latent_l2=self.latent_l2,
        )
        # ── Warm-up: build all layers ─────────────────────────────────────────
        # TransformerEncoder contains a tf.keras.Sequential (the feed-forward
        # sub-network). If encode() never calls self.trans (use_transformer=False)
        # or self.lstm (use_bilstm=False), those sub-networks remain unbuilt.
        # Keras raises "weights for model sequential have not yet been created"
        # when it inspects model.weights during fit(). Build both branches now
        # unconditionally so all weights exist before training starts.
        _xd = tf.zeros([2, self.past_len, self.d_model], dtype=tf.float32)
        _ = net.trans(net.pe(_xd), training=False)   # builds TransformerEncoder.ff
        _ = net.lstm(_xd, training=False)             # builds Bidirectional LSTM

        # ── Freeze unused branches ────────────────────────────────────────────
        # Setting trainable=False on layers that encode() will never call
        # removes their variables from optimizer tracking. Without this, the
        # Adam optimizer warns "Gradients do not exist for variables in
        # transformer_encoder/multi_head_attention/..." for every training step.
        # Freeze BEFORE compile() so the optimizer only sees active variables.
        if not self._use_transformer:
            net.pe.trainable     = False
            net.trans.trainable  = False
            net.pool_t.trainable = False
        if not self._use_bilstm:
            net.lstm.trainable   = False
            net.pool_l.trainable = False

        net.compile(optimizer=optimizers.Adam(self.lr))
        return net

    def _predict_backbone_outputs(self, X, Y):
        """
        When use_refinement=False, substitute y1 for y2 so that all
        downstream scoring uses the direct forecast error, not the
        refined forecast error.
        """
        y1, xr, Z = self.network.predict(X, batch_size=256, verbose=0)
        if self._use_refinement:
            y2 = self.network.refine(Z, Y - y1).numpy()
        else:
            y2 = y1  # no refinement: direct forecast is the final prediction
        resid = Y - y2
        return y1, xr, y2, Z, resid


### Branch Variant Configurations (Expanded, B0–B9)

| Key | Name              | CNN | Transformer | BiLSTM | Refinement |
|-----|-------------------|:---:|:-----------:|:------:|:----------:|
| B0  | full              |  ✓  |      ✓      |   ✓    |     ✓      |
| B1  | no\_cnn            |  ✗  |      ✓      |   ✓    |     ✓      |
| B2  | no\_transformer    |  ✓  |      ✗      |   ✓    |     ✓      |
| B3  | no\_bilstm         |  ✓  |      ✓      |   ✗    |     ✓      |
| B4  | no\_refinement     |  ✓  |      ✓      |   ✓    |     ✗      |
| B5  | transformer\_only  |  ✗  |      ✓      |   ✗    |     ✓      |
| B6  | bilstm\_only       |  ✗  |      ✗      |   ✓    |     ✓      |
| B7  | cnn\_only          |  ✓  |      ✗      |   ✗    |     ✓      |
| B8  | cnn\_transformer   |  ✓  |      ✓      |   ✗    |     ✓      |
| B9  | cnn\_bilstm        |  ✓  |      ✗      |   ✓    |     ✓      |

**No Refinement (B4)**: `loss_w_y2=0.0` during training; uses `y1` at
inference in place of `y2`.


In [ ]:
# ============================================================
# BRANCH-LEVEL VARIANT MAP (EXPANDED — B0–B9)
# ============================================================

BRANCH_VARIANTS = {
    "B0": ("full",             dict(use_cnn=True,  use_transformer=True,  use_bilstm=True,  use_refinement=True)),
    "B1": ("no_cnn",           dict(use_cnn=False, use_transformer=True,  use_bilstm=True,  use_refinement=True)),
    "B2": ("no_transformer",   dict(use_cnn=True,  use_transformer=False, use_bilstm=True,  use_refinement=True)),
    "B3": ("no_bilstm",        dict(use_cnn=True,  use_transformer=True,  use_bilstm=False, use_refinement=True)),
    "B4": ("no_refinement",    dict(use_cnn=True,  use_transformer=True,  use_bilstm=True,  use_refinement=False)),
    "B5": ("transformer_only", dict(use_cnn=False, use_transformer=True,  use_bilstm=False, use_refinement=True)),
    "B6": ("bilstm_only",      dict(use_cnn=False, use_transformer=False, use_bilstm=True,  use_refinement=True)),
    "B7": ("cnn_only",         dict(use_cnn=True,  use_transformer=False, use_bilstm=False, use_refinement=True)),
    "B8": ("cnn_transformer",  dict(use_cnn=True,  use_transformer=True,  use_bilstm=False, use_refinement=True)),
    "B9": ("cnn_bilstm",       dict(use_cnn=True,  use_transformer=False, use_bilstm=True,  use_refinement=True)),
}

BRANCH_CHECKPOINT_FILE = "ablation_branch_expanded_checkpoint.csv"
BRANCH_RESULTS_FILE    = "ablation_branch_expanded_results.csv"
BRANCH_SUMMARY_FILE    = "ablation_branch_expanded_summary.csv"


def run_one_branch_variant(branch_flags, past_len, horizon, dim,
                            Xp_tr_contam, Yf_tr_contam, Xp, Yf):
    """
    Instantiate a BranchVariantExpanded with the given flags, fit on
    contaminated training windows, and predict on all windows.
    """
    model = BranchVariantExpanded(
        past_len=past_len,
        horizon=horizon,
        n_features=dim,
        **branch_flags,
    )
    model.fit(
        Xp_tr_contam,
        Yf_tr_contam,
        epochs=40,
        batch_size=32,
        verbose=0,
    )
    yhat, scores = model.predict(Xp, Yf, return_scores=True)
    return yhat, scores


def run_branch_ablation_study():
    """
    Run the expanded branch-level ablation study using the SAME experimental
    grid as the original pipeline-level study (DIMENSIONS, SAMPLE_SIZES,
    ANOMALIES, CONTAMINATION_RATES, N_ITER, seed scheme, contamination logic,
    evaluation).

    Results are saved to BRANCH_RESULTS_FILE and BRANCH_SUMMARY_FILE.
    Returns (df_results, df_summary).
    """
    results   = []
    completed = set()

    if os.path.exists(BRANCH_CHECKPOINT_FILE):
        try:
            ckpt = pd.read_csv(BRANCH_CHECKPOINT_FILE)
            results = ckpt.to_dict("records")
            for row in results:
                completed.add((
                    row["Variant"],
                    row["Anomaly"],
                    int(row["Iteration"]),
                    int(row["Dim"]),
                    int(row["N_Normal"]),
                    int(row["N_Shock"]),
                    round(float(row["ContamRate"]), 6),
                ))
            print(f"Resuming branch study from {BRANCH_CHECKPOINT_FILE} "
                  f"({len(results)} rows already done)")
        except Exception as e:
            print(f"Checkpoint read failed ({e}); starting fresh.")

    total_jobs = (
        len(DIMENSIONS)
        * len(SAMPLE_SIZES)
        * len(ANOMALIES)
        * len(CONTAMINATION_RATES)
        * N_ITER
        * len(BRANCH_VARIANTS)
    )

    done     = 0
    t_global = time.perf_counter()

    for dim in DIMENSIONS:
        for (n_normal, n_shock) in SAMPLE_SIZES:
            for anomaly in ANOMALIES:
                for contam_rate in CONTAMINATION_RATES:
                    for it in range(N_ITER):
                        seed = 1000 + it
                        rng  = np.random.default_rng(seed)
                        set_all_seeds(seed)

                        try:
                            X, y = generate_stock_like_data(
                                n_normal=n_normal,
                                n_shock=n_shock,
                                p=dim,
                                anomaly_type=anomaly,
                                seed=seed,
                            )

                            Xp, Yf, y_eval = make_windows(X, y, PAST_LEN, HORIZON)

                            shock_positions = np.where(y_eval == 1)[0]
                            if len(shock_positions) == 0:
                                continue

                            shock_start    = int(shock_positions[0])
                            Xp_tr          = Xp[:shock_start]
                            Yf_tr          = Yf[:shock_start]

                            if len(Xp_tr) < 20:
                                continue

                            Xp_tr_contam, Yf_tr_contam = contaminate_training_data(
                                Xp_tr, Yf_tr, contam_rate, rng
                            )

                            base_result = dict(
                                Anomaly=anomaly,
                                Iteration=it,
                                Dim=dim,
                                N_Normal=n_normal,
                                N_Shock=n_shock,
                                ContamRate=contam_rate,
                            )

                            for variant_label, (variant_name, branch_flags) in BRANCH_VARIANTS.items():
                                done += 1
                                elapsed = time.perf_counter() - t_global
                                print(
                                    f"[{done}/{total_jobs}] "
                                    f"{variant_label} | {variant_name} | "
                                    f"anomaly={anomaly} | iter={it} | "
                                    f"elapsed={elapsed:.1f}s"
                                )

                                key = (
                                    variant_label,
                                    anomaly,
                                    it,
                                    dim,
                                    n_normal,
                                    n_shock,
                                    round(float(contam_rate), 6),
                                )
                                if key in completed:
                                    continue

                                try:
                                    tf.keras.backend.clear_session()
                                    gc.collect()
                                    set_all_seeds(seed)

                                    with Timer() as t:
                                        yhat, scores = run_one_branch_variant(
                                            branch_flags,
                                            PAST_LEN,
                                            HORIZON,
                                            dim,
                                            Xp_tr_contam,
                                            Yf_tr_contam,
                                            Xp,
                                            Yf,
                                        )

                                    p, r, f1, au, fpr = eval_metrics(
                                        y_eval, yhat, scores
                                    )

                                    row = {
                                        **base_result,
                                        "Variant":     variant_label,
                                        "VariantName": variant_name,
                                        "Precision":   p,
                                        "Recall":      r,
                                        "F1":          f1,
                                        "AUROC":       au,
                                        "FPR":         fpr,
                                        "Time":        t.elapsed,
                                    }

                                    results.append(row)
                                    completed.add(key)
                                    save_checkpoint_atomic(results, BRANCH_CHECKPOINT_FILE)

                                except Exception as e:
                                    print(f"  {variant_label} ERROR: {e}")

                        except Exception as e:
                            print(f"  Scenario ERROR: {e}")

    df = pd.DataFrame(results)
    df.to_csv(BRANCH_RESULTS_FILE, index=False)

    summary = (
        df.groupby(["Variant", "VariantName"], as_index=False)[
            ["Precision", "Recall", "F1", "AUROC", "FPR", "Time"]
        ]
        .mean()
        .sort_values("Variant")
        .reset_index(drop=True)
    )
    summary.to_csv(BRANCH_SUMMARY_FILE, index=False)

    print("\nExpanded branch-level ablation study complete.")
    print(f"Rows: {len(df)}")
    print(f"Results saved to : {BRANCH_RESULTS_FILE}")
    print(f"Summary saved to : {BRANCH_SUMMARY_FILE}")

    return df, summary


In [ ]:
# ============================================================
# RUN THE EXPANDED BRANCH-LEVEL STUDY
# ============================================================
# Checkpoint recovery is built in: re-running this cell after
# an interruption will skip already-completed rows.
# ============================================================

df_branch, df_branch_summary = run_branch_ablation_study()


## Branch-Level Results (Expanded, B0–B9)

Mean Precision, Recall, F1, AUROC, and FPR across all
anomaly types, contamination rates, and iterations.


In [ ]:
# Load from file if not already in memory (e.g., after kernel restart)
import os, pandas as pd
if "df_branch_summary" not in dir():
    if os.path.exists(BRANCH_SUMMARY_FILE):
        df_branch_summary = pd.read_csv(BRANCH_SUMMARY_FILE)
        print(f"Loaded summary from {BRANCH_SUMMARY_FILE}")
    else:
        print("Branch study has not been run yet in this session.")
        df_branch_summary = None

if df_branch_summary is not None:
    display(df_branch_summary.round(4))


## Combined Comparison: Pipeline-Level (A0–A4) vs Branch-Level (B0–B9)

The cell below merges both result files for a side-by-side view sorted by F1.


In [ ]:
import os, pandas as pd

_pipe_path   = RESULTS_FILE          # defined in Section 1
_branch_path = BRANCH_RESULTS_FILE   # defined in Section 2

combined_parts = []

if os.path.exists(_pipe_path):
    df_pipe = pd.read_csv(_pipe_path)
    df_pipe["Family"] = "pipeline"
    combined_parts.append(df_pipe)

if os.path.exists(_branch_path):
    df_br = pd.read_csv(_branch_path)
    df_br["Family"] = "branch"
    combined_parts.append(df_br)

if combined_parts:
    df_combined = pd.concat(combined_parts, ignore_index=True)
    combined_summary = (
        df_combined.groupby(["Family", "Variant", "VariantName"], as_index=False)[
            ["Precision", "Recall", "F1", "AUROC", "FPR"]
        ]
        .mean()
        .sort_values("F1", ascending=False)
        .reset_index(drop=True)
    )
    display(combined_summary.round(4))
else:
    print("Neither result file found. Run both study cells first.")
